In [23]:
import numpy as np
import pandas as pd

In [24]:
csv_path = "../data/real_bitcoin_blocks_raw.csv"
df = pd.read_csv(csv_path)
df

,number,timestamp,size,transaction_count,bits,block_number,total_output_satoshis,total_output_satoshis_excl_coinbase,total_fee_satoshis,tx_count_check,difficulty
0,0,2009-01-03 18:15:05+00:00,285,1,1d00ffff,0,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00
1,1,2009-01-09 02:54:25+00:00,215,1,1d00ffff,1,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00
2,2,2009-01-09 02:55:44+00:00,215,1,1d00ffff,2,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00
3,3,2009-01-09 03:02:53+00:00,215,1,1d00ffff,3,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00
4,4,2009-01-09 03:16:28+00:00,215,1,1d00ffff,4,5.000000e+09,0.000000e+00,0.0,1,1.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...
810904,810904,2023-10-06 12:36:45+00:00,1471359,2641,1704e90f,810904,7.607876e+11,7.601375e+11,25173010.0,2641,5.732151e+13
810905,810905,2023-10-06 12:45:49+00:00,1427754,2288,1704e90f,810905,1.161340e+12,1.160692e+12,23663392.0,2288,5.732151e+13
810906,810906,2023-10-06 12:50:15+00:00,1612966,2016,1704e90f,810906,4.643628e+11,4.637224e+11,15483511.0,2016,5.732151e+13
810907,810907,2023-10-06 13:37:00+00:00,1486563,3496,1704e90f,810907,2.469789e+12,2.469101e+12,63223285.0,3496,5.732151e+13


## Replicate Original Method

In [25]:
# Rename Columns
df = df.rename(columns={
    'number': 'block_id',
    'transaction_count': 'n_transactions',
    'total_output_satoshis_excl_coinbase': 'transaction_volume',
})

# Drop Unwanted Columns
df = df.drop(columns=['total_output_satoshis', 'total_fee_satoshis', 'block_number', 'tx_count_check', 'bits'])

# Convert satoshis to BTC and rename
satoshi_columns = ['transaction_volume']
df[satoshi_columns] = df[satoshi_columns] / 1e8

# Round-robin miner assignment
df['miner_id'] = df['block_id'] % 100

# Calculate fee proxy as n_transactions * transaction_volume
df['fee_proxy'] = df['n_transactions'] * df['transaction_volume']
df

,block_id,timestamp,size,n_transactions,transaction_volume,difficulty,miner_id,fee_proxy
0,0,2009-01-03 18:15:05+00:00,285,1,0.000000,1.000000e+00,0,0.000000e+00
1,1,2009-01-09 02:54:25+00:00,215,1,0.000000,1.000000e+00,1,0.000000e+00
2,2,2009-01-09 02:55:44+00:00,215,1,0.000000,1.000000e+00,2,0.000000e+00
3,3,2009-01-09 03:02:53+00:00,215,1,0.000000,1.000000e+00,3,0.000000e+00
4,4,2009-01-09 03:16:28+00:00,215,1,0.000000,1.000000e+00,4,0.000000e+00
...,...,...,...,...,...,...,...,...
810904,810904,2023-10-06 12:36:45+00:00,1471359,2641,7601.374620,5.732151e+13,4,2.007523e+07
810905,810905,2023-10-06 12:45:49+00:00,1427754,2288,11606.915823,5.732151e+13,5,2.655662e+07
810906,810906,2023-10-06 12:50:15+00:00,1612966,2016,4637.223611,5.732151e+13,6,9.348643e+06
810907,810907,2023-10-06 13:37:00+00:00,1486563,3496,24691.006147,5.732151e+13,7,8.631976e+07


In [26]:
miner_df = df.groupby('miner_id').agg(
    blocks_mined=('block_id', 'count'),
    avg_transactions=('n_transactions', 'mean'),
    avg_volume=('transaction_volume', 'mean'),
    avg_fee=('fee_proxy', 'mean'),
    fee_volatility=('fee_proxy', 'std'),
    avg_block_size=('size', 'mean'),
    difficulty=('difficulty', 'mean'),
    profitability=('fee_proxy', 'sum'),
    last_block_id=('block_id', 'max')
).reset_index()

miner_df['profitability'] = miner_df['profitability'] / (miner_df['blocks_mined'] + 1)
miner_df['age'] = df['block_id'].max() - miner_df['last_block_id']
miner_df['age'].describe()



count    100.000000
mean      49.500000
std       29.011492
min        0.000000
25%       24.750000
50%       49.500000
75%       74.250000
max       99.000000
Name: age, dtype: float64

In [27]:
# added very small number (1 * 10^-9) to prevent division by zero
# if any miner has avg_volume of 0
miner_df['efficiency'] = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1e-9)
miner_df.sort_values(by='efficiency', ascending=False) # show by descending efficiency


,miner_id,blocks_mined,avg_transactions,avg_volume,avg_fee,fee_volatility,avg_block_size,difficulty,profitability,last_block_id,age,efficiency
85,85,8109,1114.995067,9702.485099,1.739422e+07,4.005008e+07,637739.727217,7.823948e+12,1.739207e+07,810885,23,0.114919
84,84,8109,1115.961154,9726.562350,1.733146e+07,4.070700e+07,637289.865705,7.823948e+12,1.732932e+07,810884,24,0.114733
93,93,8109,1120.746331,9824.680572,1.788872e+07,4.281280e+07,637021.875200,7.824388e+12,1.788652e+07,810893,15,0.114075
80,80,8109,1120.253545,9883.270474,1.788742e+07,4.863221e+07,636271.152300,7.823454e+12,1.788521e+07,810880,28,0.113348
43,43,8109,1110.076952,9844.063117,1.779357e+07,5.047661e+07,634776.757800,7.821658e+12,1.779138e+07,810843,65,0.112766
...,...,...,...,...,...,...,...,...,...,...,...,...
47,47,8109,1121.243187,11055.175842,2.002822e+07,9.189077e+07,641939.092983,7.821688e+12,2.002575e+07,810847,61,0.101422
36,36,8109,1112.974226,10983.632536,2.006747e+07,1.665341e+08,636392.218276,7.821376e+12,2.006499e+07,810836,72,0.101330
48,48,8109,1109.005673,11155.246166,2.022493e+07,1.254197e+08,635688.036873,7.821693e+12,2.022243e+07,810848,60,0.099416
19,19,8109,1119.332963,11277.975085,1.873538e+07,5.464729e+07,637714.711555,7.818680e+12,1.873307e+07,810819,89,0.099249


In [28]:
# Calculate median efficiency
median_efficiency = miner_df['efficiency'].median()
print(f"Median Efficiency = {median_efficiency}")
# label miner as 1 if its efficiency is more than median, else 0
miner_df['label'] = (miner_df['efficiency'] > median_efficiency).astype(int)
miner_df[['efficiency', 'label']].head(10)

Median Efficiency = 0.10740506578081713


,efficiency,label
0,0.102810,0
1,0.102587,0
2,0.104470,0
3,0.106871,0
4,0.098501,0
5,0.103728,0
6,0.112763,1
7,0.104542,0
8,0.109948,1
9,0.106491,0


In [29]:
miner_df['label'].value_counts()

label
0    50
1    50
Name: count, dtype: int64

In [30]:
# Train and Test

from sklearn.model_selection import train_test_split

feature_columns = ['blocks_mined', 'avg_transactions', 'avg_volume', 
                   'avg_fee', 'fee_volatility', 'avg_block_size', 
                   'difficulty', 'profitability', 'age']

X = miner_df[feature_columns]
y = miner_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)

(80, 9)
(20, 9)


In [31]:
# Scale data

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# mean is close to 0 and std is 1 since we scaled it
print(f"Training Mean: {X_train_scaled.mean(axis=0)}")
print(f"Training Std: {X_train_scaled.std(axis=0)}\n")

# test data has different mean and std
# since we transformed it on the mean and std of the training data
print(f"Testing Mean: {X_test_scaled.mean(axis=0)}")
print(f"Testing Std: {X_test_scaled.std(axis=0)}")

Training Mean: [-1.21262722e-12 -2.43138842e-14  2.57016630e-15  7.88258347e-16
 -3.49720253e-16  2.27873276e-14  3.02138869e-13  1.49880108e-15
  6.10622664e-17]
Training Std: [1. 1. 1. 1. 1. 1. 1. 1. 1.]

Testing Mean: [-0.16666667 -0.3229483   0.16215183  0.38260684  0.29519136 -0.18082125
 -0.37581756  0.3826068   0.42434287]
Testing Std: [0.72648316 0.84909753 0.83748893 1.0746459  1.12235181 0.71508199
 1.01004659 1.07464583 1.00590583]


In [32]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16, 8), 
    activation='relu', 
    solver='adam', 
    alpha=0.001, 
    random_state=42)

print(model)

MLPClassifier(alpha=0.001, hidden_layer_sizes=(64, 32, 16, 8), random_state=42)


In [33]:
# Fit the model (Training)
model.fit(X_train_scaled, y_train)

# Test the accuracy of the model
# Ethan's accuracy:
# Test Accuracy = 95%
# 5-fold Cross Validation Accuracy = 48.75%
y_prediction = model.predict(X_test_scaled)
print(y_prediction)

[1 1 0 1 0 0 0 0 0 0 0 1 0 0 0 1 1 0 1 1]


In [34]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score


test_accuracy = accuracy_score(y_test, y_prediction)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)

print(test_accuracy)
print(cv_scores)
print(cv_scores.mean())

0.9
[0.875  0.875  1.     0.9375 1.    ]
0.9375


In [35]:
# High cv score found? should be closer to Ethan's Accuracy
miner_df[['avg_transactions', 'avg_volume']].corrwith(miner_df['efficiency']) 

avg_transactions   -0.000700
avg_volume         -0.982983
dtype: float64

## Modified Method with Fees

In [36]:
df = pd.read_csv(csv_path)

# Rename Columns
df = df.rename(columns={
    'number': 'block_id',
    'transaction_count': 'n_transactions',
#    'total_output_satoshis': 'total_output', # This row is inclusive of the coinbase transaction
    'total_output_satoshis_excl_coinbase': 'transaction_volume',
    'total_fee_satoshis': 'fee_proxy'
})

# Drop Extra Columns
df = df.drop(columns=['total_output_satoshis', 'block_number', 'tx_count_check', 'bits'])

# Round-robin miner assignment
df['miner_id'] = df['block_id'] % 100

# Convert satoshis to BTC
satoshi_columns = ['transaction_volume', 'fee_proxy']
df[satoshi_columns] = df[satoshi_columns] / 1e8
df

,block_id,timestamp,size,n_transactions,transaction_volume,fee_proxy,difficulty,miner_id
0,0,2009-01-03 18:15:05+00:00,285,1,0.000000,0.000000,1.000000e+00,0
1,1,2009-01-09 02:54:25+00:00,215,1,0.000000,0.000000,1.000000e+00,1
2,2,2009-01-09 02:55:44+00:00,215,1,0.000000,0.000000,1.000000e+00,2
3,3,2009-01-09 03:02:53+00:00,215,1,0.000000,0.000000,1.000000e+00,3
4,4,2009-01-09 03:16:28+00:00,215,1,0.000000,0.000000,1.000000e+00,4
...,...,...,...,...,...,...,...,...
810904,810904,2023-10-06 12:36:45+00:00,1471359,2641,7601.374620,0.251730,5.732151e+13,4
810905,810905,2023-10-06 12:45:49+00:00,1427754,2288,11606.915823,0.236634,5.732151e+13,5
810906,810906,2023-10-06 12:50:15+00:00,1612966,2016,4637.223611,0.154835,5.732151e+13,6
810907,810907,2023-10-06 13:37:00+00:00,1486563,3496,24691.006147,0.632233,5.732151e+13,7


In [37]:
miner_df = df.groupby('miner_id').agg(
    blocks_mined=('block_id', 'count'),
    avg_transactions=('n_transactions', 'mean'),
    avg_volume=('transaction_volume', 'mean'),
    avg_fee=('fee_proxy', 'mean'),
    fee_volatility=('fee_proxy', 'std'),
    avg_block_size=('size', 'mean'),
    difficulty=('difficulty', 'mean'),
    profitability=('fee_proxy', 'sum'),
    last_block_id=('block_id', 'max')   # last block a mined by a miner
).reset_index()

# convert profitability from total sum to avg profitability
miner_df['profitability'] = miner_df['profitability'] / (miner_df['blocks_mined'] + 1)
miner_df['age'] = df['block_id'].max() - miner_df['last_block_id']
miner_df.head(20)
miner_df['age'].describe()

count    100.000000
mean      49.500000
std       29.011492
min        0.000000
25%       24.750000
50%       49.500000
75%       74.250000
max       99.000000
Name: age, dtype: float64

In [38]:
# added very small number (1 * 10^-9) to prevent division by zero
# if any miner has avg_volume of 0
miner_df['efficiency'] = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1e-9)
miner_df.sort_values(by='efficiency', ascending=False).head(10) # show by descending efficiency

,miner_id,blocks_mined,avg_transactions,avg_volume,avg_fee,fee_volatility,avg_block_size,difficulty,profitability,last_block_id,age,efficiency
85,85,8109,1114.995067,9702.485099,0.330160,0.698048,637739.727217,7.823948e+12,0.330119,810885,23,0.114919
84,84,8109,1115.961154,9726.562350,0.350161,1.530179,637289.865705,7.823948e+12,0.350117,810884,24,0.114733
93,93,8109,1120.746331,9824.680572,0.340470,0.786819,637021.875200,7.824388e+12,0.340428,810893,15,0.114075
80,80,8109,1120.253545,9883.270474,0.328159,0.690659,636271.152300,7.823454e+12,0.328118,810880,28,0.113348
43,43,8109,1110.076952,9844.063117,0.331859,0.916274,634776.757800,7.821658e+12,0.331818,810843,65,0.112766
6,6,8110,1117.373490,9909.006859,0.333799,0.706850,636283.305795,7.823309e+12,0.333758,810906,2,0.112763
77,77,8109,1119.855963,9953.447977,0.343367,1.185601,640469.891109,7.823423e+12,0.343325,810877,31,0.112509
96,96,8109,1100.668270,9796.187177,0.336090,0.735388,628383.120237,7.824585e+12,0.336049,810896,12,0.112357
71,71,8109,1111.978296,9917.650844,0.331271,0.707519,638115.643976,7.822642e+12,0.331231,810871,37,0.112121
92,92,8109,1117.975459,9984.001458,0.329979,0.706376,633749.164385,7.824388e+12,0.329938,810892,16,0.111977


In [39]:
# Calculate median efficiency
median_efficiency = miner_df['efficiency'].median()
print(f"Median Efficiency = {median_efficiency}")
# label miner as 1 if its efficiency is more than median, else 0
miner_df['label'] = (miner_df['efficiency'] > median_efficiency).astype(int)
miner_df[['efficiency', 'label']].head(10)
miner_df['label'].value_counts()

Median Efficiency = 0.10740506578081713


label
0    50
1    50
Name: count, dtype: int64

In [40]:
# Train and Test

feature_columns = ['blocks_mined', 'avg_transactions', 'avg_volume', 
                   'avg_fee', 'fee_volatility', 'avg_block_size', 
                   'difficulty', 'profitability', 'age']

X = miner_df[feature_columns]
y = miner_df['label']

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale data using normalization so that different features
# with huge numbers and small numbers have equal importance

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [42]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(hidden_layer_sizes=(64, 32, 16, 8), activation='relu', solver='adam', alpha=0.001, random_state=42)

print(model)

# Fit the model (Training)
model.fit(X_train_scaled, y_train)

MLPClassifier(alpha=0.001, hidden_layer_sizes=(64, 32, 16, 8), random_state=42)


,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.001
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [43]:
# Test the accuracy of the model
# Ethan's accuracy:
# Test Accuracy = 95%
# 5-fold Cross Validation Accuracy = 48.75%
y_prediction = model.predict(X_test_scaled)
print(y_prediction)

from sklearn.model_selection import cross_val_score


cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
print(cv_scores)
print(cv_scores.mean())

[1 1 0 1 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 1]
[0.9375 0.875  0.9375 0.875  0.9375]
0.9125


In [44]:
# High cv score found? should be closer to Ethan's Accuracy
miner_df[['avg_transactions', 'avg_volume']].corrwith(miner_df['efficiency']) 

avg_transactions   -0.000700
avg_volume         -0.982983
dtype: float64

In [45]:
# TODO: Combine both made up proxy fee and actual proxy fee into modular code that allows for switching instead of rewriting everything again